In [0]:
dbutils.library.restartPython()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Model Training — Azure ML MLflow tracking
# MAGIC Same pipeline as `databricks_mlflow_training.py`, but points MLflow
# MAGIC at your **Azure ML workspace** instead of Databricks' built-in tracking
# MAGIC server. This registers the model directly into the Azure ML Model
# MAGIC Registry, which is where `deploy.py` looks for it.
# MAGIC
# MAGIC `azureml-mlflow` authenticates via `DefaultAzureCredential`, which reads
# MAGIC `AZURE_CLIENT_ID` / `AZURE_CLIENT_SECRET` / `AZURE_TENANT_ID` from the
# MAGIC process environment automatically — no explicit credential object needs
# MAGIC to be constructed in this notebook.
# MAGIC
# MAGIC **Before running this notebook, you need:**
# MAGIC 1. Libraries `azureml-mlflow` and `azure-identity` installed (Cluster → Libraries tab)
# MAGIC 2. `AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET`, `AZURE_TENANT_ID` set as cluster
# MAGIC    environment variables (Cluster → Edit → Advanced Options → Spark →
# MAGIC    Environment Variables), using the Azure ML service principal's credentials,
# MAGIC    followed by a cluster restart for the change to take effect.

# COMMAND ----------

storage_account = "enterprisemlopssa"  # <-- change if yours differs
processed_path = f"abfss://processed@{storage_account}.dfs.core.windows.net/asset_features"

# COMMAND ----------

# MAGIC %md
# MAGIC ## Verify the service principal credentials are set as cluster environment variables
# MAGIC This cluster has `AZURE_CLIENT_ID` / `AZURE_CLIENT_SECRET` / `AZURE_TENANT_ID`
# MAGIC set under Compute -> Edit -> Advanced Options -> Spark -> Environment
# MAGIC Variables. `DefaultAzureCredential` (used internally by azureml-mlflow)
# MAGIC reads these automatically from the process environment -- this cell just
# MAGIC checks they're actually present before wasting a training run.

# COMMAND ----------

import os

missing = [v for v in ("AZURE_CLIENT_ID", "AZURE_CLIENT_SECRET", "AZURE_TENANT_ID") if not os.environ.get(v)]
if missing:
    raise EnvironmentError(
        f"Missing cluster environment variable(s): {missing}. Set these under "
        "Cluster -> Edit -> Advanced Options -> Spark -> Environment Variables "
        "using the Azure ML service principal's credentials, then restart the "
        "cluster (cluster env var changes require a restart to take effect)."
    )

print("Azure ML service principal environment variables are present.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Verify the required libraries are actually installed before wasting a training run

# COMMAND ----------

try:
    import azureml.mlflow  # noqa: F401
except ImportError as exc:
    raise ImportError(
        "azureml-mlflow is not installed on this cluster. Add it under "
        "Cluster -> Libraries -> Install New -> PyPI -> 'azureml-mlflow', "
        "then 'azure-identity' the same way, and restart the cluster."
    ) from exc

print("Required libraries are installed.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Paste your Azure ML MLflow tracking URI
# MAGIC Get it from your terminal (not this notebook):
# MAGIC ```bash
# MAGIC az ml workspace show --name mlw-enterprise-mlops --resource-group rg-enterprise-mlops --query mlflow_tracking_uri -o tsv
# MAGIC ```
# MAGIC Using a widget here (instead of hardcoding it in the cell) means you're not
# MAGIC committing a workspace-specific URI into notebook source if this gets
# MAGIC checked back into the repo via Databricks Repos.

# COMMAND ----------

dbutils.widgets.text("azure_ml_tracking_uri", "", "Azure ML MLflow tracking URI")
azure_ml_tracking_uri = dbutils.widgets.get("azure_ml_tracking_uri")

if not azure_ml_tracking_uri:
    raise ValueError(
        "azure_ml_tracking_uri widget is empty. Fill it in at the top of this "
        "notebook (or via Run > Edit Parameters) before running further cells."
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## Import the training logic straight from the repo

# COMMAND ----------

import sys

notebook_dir = os.getcwd()
repo_root = os.path.abspath(os.path.join(notebook_dir, ".."))
src_dir = os.path.join(repo_root, "src")
sys.path.insert(0, src_dir)

from train import train  # noqa: E402

# COMMAND ----------

# MAGIC %md
# MAGIC ## Bridge: Delta (Spark) -> local Parquet (pandas)

# COMMAND ----------

features_sdf = spark.read.format("delta").load(processed_path)
features_pdf = features_sdf.toPandas()
print(f"Loaded {len(features_pdf):,} rows from {processed_path}")

local_features_dir = "/local_disk0/tmp/enterprise-mlops"  # local disk on the cluster node, not DBFS
os.makedirs(local_features_dir, exist_ok=True)
local_features_path = f"{local_features_dir}/asset_features.parquet"
features_pdf.to_parquet(local_features_path)
print(f"Wrote local copy for training -> {local_features_path}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Point MLflow at Azure ML
# MAGIC With `AZURE_CLIENT_ID` / `AZURE_CLIENT_SECRET` / `AZURE_TENANT_ID` now set
# MAGIC in this session's environment, `DefaultAzureCredential` (used internally
# MAGIC by the azureml-mlflow plugin) picks them up automatically -- no explicit
# MAGIC credential object needs to be constructed here.
# MAGIC
# MAGIC **Both `set_tracking_uri` AND `set_registry_uri` are needed.** MLflow
# MAGIC treats "where runs/metrics are logged" and "where models get registered"
# MAGIC as two separate destinations. On a Unity Catalog-enabled workspace,
# MAGIC the registry silently defaults to Databricks' own Unity Catalog
# MAGIC registry (`databricks-uc`) unless you override it explicitly -- which is
# MAGIC exactly what caused the earlier "model must be logged with a model
# MAGIC signature... Unity Catalog" error: registration was quietly going to UC
# MAGIC instead of Azure ML.

# COMMAND ----------

import mlflow

mlflow.set_tracking_uri(azure_ml_tracking_uri)
mlflow.set_registry_uri(azure_ml_tracking_uri)
mlflow.set_experiment("enterprise-asset-risk")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Train
# MAGIC `train()` calls `mlflow.sklearn.log_model(..., registered_model_name=...)`
# MAGIC internally — with the tracking URI pointed at Azure ML, this registers
# MAGIC straight into the Azure ML workspace's Model Registry.

# COMMAND ----------

metrics, run_id = train(
    features_path=local_features_path,
    model_type="random_forest",
    model_dir="/local_disk0/tmp/enterprise-mlops-models",  # local disk on the cluster node, not DBFS; unused when MLflow is available
    register_from_local_file=True,  # works around Azure ML's run_id-length bug in CreateModelVersion
)

print("Run ID:", run_id)
print("Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Where to look next
# MAGIC Azure ML Studio → your workspace → **Models** → `asset-failure-risk-classifier`.
# MAGIC Note the version number shown there — you'll need it for `deploy.py --model-version`.